# EchoFactory - VALVE: Evaluasi & Export ONNX (KNN Scoring)


In [ ]:
import os, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MACHINE_TYPE = 'valve'
FEAT_DIR = '/kaggle/working/features'
model_path = f'/kaggle/working/stgram_mfn_{MACHINE_TYPE}.pt'


In [ ]:
class ConvBNPReLU(nn.Module):
    def __init__(self, ic, oc, k=3, s=1, p=1, g=1):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(ic, oc, k, s, p, groups=g, bias=False), nn.BatchNorm2d(oc), nn.PReLU(oc))
    def forward(self, x): return self.net(x)

class DepthwiseSep(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        self.net = nn.Sequential(ConvBNPReLU(ic, ic, s=s, g=ic), ConvBNPReLU(ic, oc, k=1, p=0))
    def forward(self, x): return self.net(x)

class MobileFaceNet(nn.Module):
    def __init__(self, ed=256):
        super().__init__()
        self.enc = nn.Sequential(ConvBNPReLU(1, 32, s=2), DepthwiseSep(32, 64), DepthwiseSep(64, 128, s=2), DepthwiseSep(128, 128), DepthwiseSep(128, 256, s=2), DepthwiseSep(256, 256), DepthwiseSep(256, 512, s=2), nn.AdaptiveAvgPool2d(1))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(512, ed), nn.BatchNorm1d(ed))
    def forward(self, x): return self.head(self.enc(x))

class STgramMFN(nn.Module):
    def __init__(self, nc, ed=256):
        super().__init__()
        self.mel, self.tgram = MobileFaceNet(ed), MobileFaceNet(ed)
        self.fuse = nn.Sequential(nn.Linear(ed * 2, ed), nn.BatchNorm1d(ed), nn.PReLU(ed))
    def forward(self, mel, tg):
        return F.normalize(self.fuse(torch.cat([self.mel(mel), self.tgram(tg)], dim=1)), dim=1)


In [ ]:
@torch.no_grad()
def get_embeddings_with_labels(model, feat_dir, machine, cond):
    data = torch.load(os.path.join(feat_dir, f'{machine}_{cond}.pt'))
    mel_feats = data['features'][:, 0:1]
    tg_feats  = data['features'][:, 1:2]
    labels    = data['labels']
    dl = DataLoader(TensorDataset(mel_feats, tg_feats, labels), batch_size=128, shuffle=False)
    model.eval()
    embs, lbls = [], []
    for mb, tb, lb in dl:
        embs.append(model(mb.to(device), tb.to(device)).cpu())
        lbls.append(lb)
    return torch.cat(embs).numpy(), torch.cat(lbls).numpy()

def compute_knn_anomaly_scores(normal_embs, normal_labels, test_embs, test_labels, k=5):
    unique_ids = np.unique(normal_labels)
    knn_models = {}
    for uid in unique_ids:
        mask = (normal_labels == uid)
        knn = NearestNeighbors(n_neighbors=k, metric='cosine')
        knn.fit(normal_embs[mask])
        knn_models[int(uid)] = knn
    
    scores = []
    for i in range(len(test_embs)):
        emb = test_embs[i:i+1]
        lid = int(test_labels[i])
        if lid in knn_models:
            knn = knn_models[lid]
        else:
            knn = list(knn_models.values())[0]
        distances, _ = knn.kneighbors(emb)
        scores.append(distances.mean())
    return np.array(scores)

def compute_pauc(y_true, scores, max_fpr=0.1):
    fpr, tpr, _ = roc_curve(y_true, scores)
    mask = fpr <= max_fpr
    return float(np.trapz(tpr[mask], fpr[mask]) / max_fpr)


In [ ]:
if os.path.exists(model_path):
    ck = torch.load(model_path, map_location=device)
    model = STgramMFN(ck['n_classes'], ck['embed_dim']).to(device)
    model.load_state_dict(ck['model_state'])
    model.eval()
    
    ne, n_lbls = get_embeddings_with_labels(model, FEAT_DIR, MACHINE_TYPE, 'normal')
    ae, a_lbls = get_embeddings_with_labels(model, FEAT_DIR, MACHINE_TYPE, 'abnormal')
    
    ns = compute_knn_anomaly_scores(ne, n_lbls, ne, n_lbls, k=5)
    as_ = compute_knn_anomaly_scores(ne, n_lbls, ae, a_lbls, k=5)
    
    scores = np.concatenate([ns, as_])
    labels = np.concatenate([np.zeros(len(ns)), np.ones(len(as_))])
    
    auc_val = roc_auc_score(labels, scores)
    pauc_val = compute_pauc(labels, scores)
    
    fpr, tpr, thr = roc_curve(labels, scores)
    best_thr = float(thr[np.argmax(tpr - fpr)])
    
    print(f'=== EVALUASI DENGAN KNN (k=5): {MACHINE_TYPE.upper()} ===')
    print(f'🚀 AUC  Score : {auc_val:.4f}')
    print(f'🚀 pAUC Score : {pauc_val:.4f}')
    print(f'🚀 Threshold  : {best_thr:.4f}')
    
    # ROC Plot
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, 'b-', lw=2, label=f'AUC={auc_val:.4f}')
    plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    plt.fill_between(fpr[fpr <= 0.1], tpr[fpr <= 0.1], alpha=0.2, color='orange', label=f'pAUC={pauc_val:.4f}')
    plt.title(f'ROC Curve (KNN) - {MACHINE_TYPE.upper()}', fontweight='bold')
    plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/roc_{MACHINE_TYPE}.png', dpi=150)
    plt.show()
    
    # Export ONNX
    m_cpu = STgramMFN(ck['n_classes'], ck['embed_dim'])
    m_cpu.load_state_dict(ck['model_state']); m_cpu.eval()
    onnx_path = f'/kaggle/working/stgram_mfn_{MACHINE_TYPE}.onnx'
    torch.onnx.export(m_cpu, (torch.randn(1, 1, 128, 128), torch.randn(1, 1, 128, 128)), onnx_path, input_names=['mel', 'tgram'], output_names=['embedding'])
    print(f'ONNX Exported: {onnx_path}')
